# Tema 2 · Regresión — Modelo de Liquidez
## Ejercicio del sábado (plantilla) · Dataset: **Company Bankruptcy (Taiwan)**

**Caso de negocio.** Dispones de **95 ratios financieros** de ~6.800 empresas
taiwanesas. Tu objetivo es construir un modelo de regresión que **estime la
liquidez** de cada empresa, medida con el **`Current Ratio`** (variable objetivo),
a partir del resto de indicadores. Con tantas variables correlacionadas, el reto
central es **manejar la multicolinealidad** y **evitar la fuga de información**.

> **Cómo se evalúa.** Completa las celdas `# TODO:`. Las **secciones 1 y 2 ya
> están resueltas** — presta especial atención a la 2.3 (exclusión de variables
> casi-duplicadas del target), que es el punto clave de este ejercicio.

Estructura:
1. Carga y exploración ✅ *(resuelto)*
2. Limpieza: outliers del target + **exclusión de variables casi-duplicadas** ✅ *(resuelto)*
3. Selección de variables y split — **TODO**
4. Modelos: lineal + Ridge/Lasso — **TODO**
5. Evaluación (MAE, RMSE, R²) + residuos — **TODO**
6. Multicolinealidad: VIF — **TODO**
7. Interpretación de coeficientes — **TODO**
8. Conclusión de negocio — **preguntas abiertas**


## 0. Semilla personal (anti-copia)

In [ ]:
cedula = 1020304050  # <-- REEMPLAZA por tu número de cédula completo
import numpy as np
np.random.seed(cedula)
print(f"Semilla fijada en {cedula}")

## 1. Carga y exploración  ✅ *(resuelto)*

Los nombres de columna del CSV traen un espacio inicial → lo limpiamos.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

df = pd.read_csv("../../data/03-regresion/taiwan_bankruptcy.csv")
df.columns = [c.strip() for c in df.columns]
print("Dimensiones:", df.shape)
df.head()

In [ ]:
df = df.sample(frac=0.85, random_state=cedula).reset_index(drop=True)
print("Dimensiones tras el muestreo:", df.shape)

## 2. Limpieza  ✅ *(resuelto — estúdiala con atención)*

### 2.1 El target tiene outliers extremos
El `Current Ratio` en este dataset está normalizado, pero tiene un ~1% de valores
atípicos gigantescos que distorsionan cualquier regresión. Los recortamos por el
**percentil 99**.

In [ ]:
target = "Current Ratio"
print("Antes del recorte — describe del target:")
print(df[target].describe())

p99 = df[target].quantile(0.99)
df = df[df[target] <= p99].copy()
print(f"\nRecortado en p99 = {p99:.4f}. Filas restantes: {df.shape[0]}")
sns.histplot(df[target], kde=True, color="#4361ee")
plt.title("Current Ratio tras recortar outliers (p99)")
plt.show()

### 2.2 Quitar la columna de bancarrota
`Bankrupt?` es la etiqueta de otro problema (clasificación de quiebra), no un
predictor de liquidez → la descartamos.

In [ ]:
if "Bankrupt?" in df.columns:
    df = df.drop(columns=["Bankrupt?"])
print("Columnas:", df.shape[1])

### 2.3 ⚠️ EXCLUIR variables casi-duplicadas del target (fuga de información)

**Este es el punto de teoría clave del día.** Algunas columnas miden
esencialmente **lo mismo** que el Current Ratio (son una reformulación del ratio
de liquidez). Si las dejáramos como predictoras, el modelo tendría un R²
artificialmente alto pero **no aprendería nada útil**: estaría "haciendo trampa"
al ver una copia del target. Esto se llama **fuga de información (data leakage)**.

Las excluimos explícitamente:

In [ ]:
leakage_cols = [
    "Quick Ratio",
    "Quick Assets/Current Liability",
    "Cash/Current Liability",
    "Current Liability to Assets",
    "Current Liability to Current Assets",
]
present = [c for c in leakage_cols if c in df.columns]
print("Variables excluidas por fuga/colinealidad extrema con el target:")
for c in present:
    print("  -", c)
df = df.drop(columns=present)
print("\nColumnas restantes (incluye el target):", df.shape[1])

---
# A partir de aquí, completa tú el análisis (secciones 3–8)
---

## 3. Selección de variables y split  — **TODO**

- Separa `X` (todas las columnas menos el target) e `y` (`Current Ratio`).
- Haz `train_test_split` con `test_size=0.25` y `random_state=cedula`.

> Opcional: puedes quedarte con las variables más correlacionadas con el target
> para un modelo más simple; documenta tu criterio.

In [ ]:
# TODO: separa X e y, y haz el train/test split


## 4. Modelos: lineal + Ridge/Lasso  — **TODO**

Entrena al menos una **regresión lineal** (baseline) y un modelo **regularizado**
(Ridge y/o Lasso). Escala las variables dentro de un `Pipeline`
(`StandardScaler` + modelo) — con 90 predictores, la regularización importa.

In [ ]:
# TODO: crea pipelines (StandardScaler + modelo) y entrénalos


## 5. Evaluación (MAE, RMSE, R²) + residuos  — **TODO**

Compara los modelos en el test con **MAE, RMSE y R²**. Para el mejor modelo,
grafica **residuos vs. predichos** y un **QQ-plot** para revisar los supuestos.

In [ ]:
# TODO: calcula MAE/RMSE/R2 y grafica el diagnóstico de residuos


## 6. Multicolinealidad: VIF  — **TODO**

Calcula el **VIF** de las variables (o de un subconjunto). Recuerda:
**VIF > 5-10** indica multicolinealidad problemática. Con tantos ratios
financieros correlacionados, esperarás varios VIF altos.

> `from statsmodels.stats.outliers_influence import variance_inflation_factor`

In [ ]:
# TODO: calcula el VIF y muéstralo ordenado de mayor a menor


## 7. Interpretación de coeficientes  — **TODO**

Con las variables escaladas, muestra los coeficientes del modelo (Ridge/Lasso)
ordenados por magnitud. ¿Qué ratios explican mejor la liquidez?

In [ ]:
# TODO: grafica los coeficientes ordenados por magnitud


## 8. Conclusión de negocio  — **responde en texto**

Responde brevemente:

1. **¿Qué ratios muestran mayor riesgo de multicolinealidad, y cómo lo detectaste
   (VIF, heatmap de correlaciones)?**

   _(tu respuesta aquí)_

2. **Si tuvieras que reducir el modelo a solo 3 variables por simplicidad
   operativa, ¿cuáles elegirías y por qué?**

   _(tu respuesta aquí)_

3. **Explica con tus palabras por qué excluimos variables como *Quick Ratio* o
   *Cash/Current Liability* del conjunto de predictores. ¿Qué habría pasado con
   el R² si las hubiéramos dejado?**

   _(tu respuesta aquí)_
